### Descriptor for relative permittivity, centroid shift, and 5d1 Ce models

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm

from pymatgen.core import Structure
from pymatgen.core.periodic_table import Element, Specie
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.analysis.local_env import CrystalNN
from pymatgen.analysis.chemenv.coordination_environments.coordination_geometry_finder import LocalGeometryFinder
from pymatgen.transformations.advanced_transformations import OrderDisorderedStructureTransformation

from mp_api.client import MPRester
from pymatgen.io.cif import CifParser

### Cation Site Features

In [ ]:
polar_pointgroups = ["1","2","m","mm2","3","3m","4","4mm","6","6mm"] 
average_anion_polarizability = {"F": 0.634, "Cl": 2.2, "Br": 3.1, "I": 5, "O": 0.793, "S": 2.9, "Se": 3.8, "N": 1.1}
cations_of_interest = ["Na+","K+","Rb+","Cs+","Ca2+","Sr2+","Ba2+","Y3+","Ce4+","La3+","Pr3+",
                       "Nd3+","Sm3+","Eu3+","Gd3+","Tb3+","Dy3+","Er3+","Tm3+","Yb3+","Lu3+"] 

In [ ]:
def _custom_formatwarning(msg, *args, **kwargs):
    return f"{msg}\n"
warnings.formatwarning = _custom_formatwarning

def roman(number):
    num = [1, 4, 5, 9, 10, 40, 50, 90, 100, 400, 500, 900, 1000]
    sym = ["I", "IV", "V", "IX", "X", "XL", "L", "XC", "C", "CD", "D", "CM", "M"]
    i = 12
    roman_string = ""

    while number:
        div = number // num[i]
        number %= num[i]

        while div:
            roman_string += sym[i]
            div -= 1
        i -= 1

    return roman_string

### Cation Site Features

In [ ]:
def get_polyhedron_dict(structure, cation_site_index, neighbors, dopant_species="Ce3+", verbose=False):
    polyhedron_dict = {}
    cn = len(neighbors)

    structure_w_oxi = structure.copy()
    structure_w_oxi.add_oxidation_state_by_guess(max_sites=-1)  # add oxidation states to structure

    polyhedron_dict["Formula"] = structure_w_oxi.composition.reduced_formula

    polyhedron_dict["coordination_number"] = cn

    cation_specie = structure_w_oxi.sites[cation_site_index].specie
    cn_roman = roman(cn)

    def get_radius(specie, coord_num):
        try: 
            return specie.get_shannon_radius(cn=coord_num, radius_type="ionic") 
        except (KeyError, ValueError): 
            return specie.ionic_radii.get(specie.oxi_state, specie.average_ionic_radius)

    cation_ionic_radius = get_radius(cation_specie, cn_roman)
    dopant_specie_obj = Specie.from_str(dopant_species)
    dopant_ionic_radius = get_radius(dopant_specie_obj, cn_roman)

    polyhedron_dict["cation_ionic_radius"] = cation_ionic_radius  # Angstrom
    polyhedron_dict["dopant_ionic_radius"] = dopant_ionic_radius  # Angstrom
    polyhedron_dict["delta_radius"] = (
        cation_ionic_radius - dopant_ionic_radius
    )  # Angstrom

    dist_list = [n['site'].distance(structure[cation_site_index]) for n in neighbors]
    polyhedron_dict["Avg_bond_length"] = np.mean(dist_list)
    
    return polyhedron_dict

def get_chemenv_info(all_coord_envs, isite):
    coord_envs = all_coord_envs

    if isite >= len(coord_envs) or not coord_envs[isite]:
        return None 
    
    my_env = coord_envs[isite] 
    if len(my_env) > 1: 
        my_env = sorted(my_env, key=lambda x: x["ce_fraction"], reverse=True) 
        
    try: 
        res = my_env[0] 
        ce_symbol = res.get("ce_symbol")
        chemenv_CN = ce_symbol.split(":")[1] if ce_symbol else None 
        return {"chemenv_CN": chemenv_CN}
    
    except (IndexError, KeyError, AttributeError): 
        return {"chemenv_CN": chemenv_CN}
    

### Crystal Structure Features

In [ ]:
def get_struc_dict(structure, sga):
    struc_dict = {}
    conv_struc = sga.get_conventional_standard_structure()

    struc_dict["SGR No."] = sga.get_space_group_number()
    struc_dict["density"] = conv_struc.density
    struc_dict["inversion center"] = sga.is_laue()
    struc_dict["inversion center"] = int(struc_dict["inversion center"])
    struc_dict["polar axis"] = sga.get_point_group_symbol() in polar_pointgroups
    struc_dict["polar axis"] = int(struc_dict["polar axis"])
    struc_dict["a/b"] = conv_struc.lattice.a / conv_struc.lattice.b
    struc_dict["b/c"] = conv_struc.lattice.b / conv_struc.lattice.c
    struc_dict["c/a"] = conv_struc.lattice.c / conv_struc.lattice.a
    struc_dict["alpha/beta"] = conv_struc.lattice.alpha / conv_struc.lattice.beta
    struc_dict["beta/gamma"] = conv_struc.lattice.beta / conv_struc.lattice.gamma
    struc_dict["gamma/alpha"] = conv_struc.lattice.gamma / conv_struc.lattice.alpha
    struc_dict["volume_rp"] = conv_struc.volume /10**3
    struc_dict["volume_per_Z_rp"] = conv_struc.volume / conv_struc.composition\
        .get_reduced_composition_and_factor()[1] / 10**3
    struc_dict["volume_per_atom_rp"] = (conv_struc.volume / conv_struc.composition.num_atoms) / 10**3  # Angstrom^3

    structure.add_oxidation_state_by_guess(max_sites=-1)
    struc_dict["Avg. cation electronegativity"] = np.mean([
        Element(site.specie.element).X
        for site in structure.sites
        if site.specie.oxi_state >= 0
    ])
    struc_dict["Avg. anion polarizability"] = np.mean([
        average_anion_polarizability[site.specie.element.symbol]
        for site in structure.sites
        if site.specie.oxi_state < 0
    ])

    num_anions = len([site for site in structure.sites if site.specie.oxi_state < 0])
    num_cations = len([site for site in structure.sites if site.specie.oxi_state >= 0])
    struc_dict["Condensation"] = num_anions / num_cations

    return struc_dict


### Get Features from Materials Project

In [ ]:
import pandas as pd
from tqdm import tqdm
from mp_api.client import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.analysis.local_env import CrystalNN

comp_df = pd.read_excel("tester.xlsx")
target_compositions = comp_df['Composition'].unique().tolist()

materials_fields = [
    "material_id",
    "formula_pretty",
    "structure",
    "symmetry",
    "volume",
    "density"
]

Feature_dicts = []

api_key = os.environ.get("MP_API") #set your MP API as env variable

with MPRester(api_key) as mpr:
    docs = mpr.materials.summary.search(formula=target_compositions, fields=materials_fields)

    cnn = CrystalNN(cation_anion=True)
    lgf = LocalGeometryFinder()

for i, doc in enumerate(tqdm(docs)):
    try: 
        formula = doc.formula_pretty
        struc=doc.structure
        struc.add_oxidation_state_by_guess(max_sites=-1)

        all_elements_in_struc = [getattr(s.specie, "element", s.specie).symbol for s in struc.sites]
        valid_sites = [idx for idx, sym in enumerate(all_elements_in_struc) if sym in cations_of_interest]
        
        all_nn_info = cnn.get_all_nn_info(struc)
        lgf.setup_structure(structure=struc)
        all_coord_envs = lgf.compute_coordination_environments(struc)
        sga = SpacegroupAnalyzer(struc)
        sym_dataset = sga.get_symmetry_dataset()
    
        inequivalent_site_indices = set(sym_dataset["equivalent_atoms"])

        inequivalent_cation_site_indices = [
            idx for idx in inequivalent_site_indices
            if str(doc.structure.sites[idx].specie) in cations_of_interest
        ]
    
        for cation_site_index in inequivalent_cation_site_indices:
            cation_specie = doc.structure.sites[cation_site_index].specie
            Feature_dicts += [
                    {
                        "Composition": str(doc.formula_pretty),
                        "Database IDs": doc.material_id,
                        "Central Cation": str(doc.structure.sites[cation_site_index].specie),
                        **get_polyhedron_dict(struc, cation_site_index, all_nn_info[cation_site_index]),
                        **get_chemenv_info(all_coord_envs, cation_site_index),
                        **get_struc_dict(struc, sga)
                    }
                ]    

    except IndexError:
        print(f"skipping composition {doc.formula_pretty} due to IndexError")
    except Exception as e:
        print(f"Error in {doc.formula_pretty}: {e}")

if Feature_dicts: 
    temp_df = pd.DataFrame(Feature_dicts) 
    if "chemenv_CN" in temp_df.columns:
        temp_df["chemenv_CN"] = temp_df["chemenv_CN"].fillna(temp_df["coordination_number"])

### Get Features from CIF Files

In [ ]:
from pymatgen.core import Structure
from tqdm import tqdm

def _custom_formatwarning(msg, *args, **kwargs):
    return f"{msg}\n"
warnings.formatwarning = _custom_formatwarning

cnn = CrystalNN(cation_anion=True)
lgf = LocalGeometryFinder()

cif_directory  = os.getcwd()
Feature_dicts = []

cif_files = [f for f in os.listdir(cif_directory) if f.endswith(".cif")]
print(f"total number of cif: {len(cif_files)}")

# put the cif files in the same folder where this code exist
for cif_filename in tqdm(cif_files, desc="CIF Processing"):
    cif_file_path = os.path.join(cif_directory, cif_filename)

    try:
        structure = Structure.from_file(cif_file_path, primitive=True)

        if not structure.is_ordered:
            odt = OrderDisorderedStructureTransformation()
            ordered_structures = odt.apply_transformation(structure, return_ranked_list=True)
            if ordered_structures:
                structure = ordered_structures[0]["structure"]

        structure.add_oxidation_state_by_guess(max_sites=-1)
        formula = structure.composition.reduced_formula

        sga = SpacegroupAnalyzer(structure)
        symm_dataset = sga.get_symmetry_dataset()
        inequivalent_site_indices = set(symm_dataset["equivalent_atoms"])

        all_nn_info = cnn.get_all_nn_info(structure)

        lgf.setup_structure(structure=structure)
        all_coord_envs = lgf.compute_coordination_environments(structure)

        inequivalent_cation_site_indices = []
        for i in inequivalent_site_indices:
            site = structure.sites[i]
            sp_str = site.species_string

            if sp_str in cations_of_interest:
                inequivalent_cation_site_indices.append(i)
            elif sp_str.replace('+', '') + '+' in cations_of_interest:
                inequivalent_cation_site_indices.append(i)

        if not inequivalent_cation_site_indices:
            continue
        
        for cation_site_index in inequivalent_cation_site_indices:
            Feature_dicts += [
                {
                    "Composition": formula,
                    "Database IDs": cif_filename,
                    "Central Cation": str(structure.sites[cation_site_index].specie),
                    **get_polyhedron_dict(structure, cation_site_index, all_nn_info[cation_site_index]),
                    **get_chemenv_info(all_coord_envs, cation_site_index),
                    **get_struc_dict(structure, sga)
                }
            ]
        print(f"  [Success] {cif_filename} parsed.")    

    except Exception as e:
        print(f"\n  [Error] {cif_filename}: {e}")

if Feature_dicts: 
    temp_df = pd.DataFrame(Feature_dicts) 
    if "chemenv_CN" in temp_df.columns:
        temp_df["chemenv_CN"] = temp_df["chemenv_CN"].fillna(temp_df["coordination_number"])

In [ ]:
rp_columns = [
    'Composition',
    'Database IDs',
    'Central Cation',
    'SGR No.',
    'volume_rp',
    'density',
    'a/b',
    'b/c',
    'c/a',
    'alpha/beta',
    'beta/gamma',
    'gamma/alpha',
    'inversion center',
    'polar axis',
    'volume_per_Z_rp',
    'volume_per_atom_rp'
]

cs_columns = [
    'Composition',
    'Database IDs',
    'Central Cation',
    'coordination_number',
    'cation_ionic_radius',
    'delta_radius',
    'Avg_bond_length',
    'Avg. cation electronegativity',
    'Avg. anion polarizability',
    'Condensation'
]

d1_columns = [
    'Composition',
    'Database IDs',
    'Central Cation',
    'coordination_number',
    'cation_ionic_radius',
    'dopant_ionic_radius',
    'chemenv_CN',
    'SGR No.',
]

RP_df = temp_df[rp_columns]
CS_df = temp_df[cs_columns]
d1_df = temp_df[d1_columns]

### Get the Relative Permittivity

In [ ]:
from pymatgen.core.composition import Composition

permittivity_df = RP_df[["Composition"]]

class Vectorize_Formula:
    def __init__(self):
        elem_dict = pd.read_excel(r'elements_rp_new.xlsx') # CHECK NAME OF FILE
        self.element_df = pd.DataFrame(elem_dict)
        self.element_df.set_index('Element',inplace=True)
        self.column_names = []
        for string in ['avg','diff','max','min','std']:
            for column_name in list(self.element_df.columns.values):
                self.column_names.append(f'{string}_{column_name}')

    def get_features(self, formula):
        try:
            fractional_composition = Composition(formula).fractional_composition.as_dict()
            avg_feature = np.zeros(len(self.element_df.columns))

            for key in fractional_composition:
                if key not in self.element_df.index:
                    print('The element:', key, 'from formula', formula,'is not currently supported in our database')
                    return np.full(len(self.element_df.colums)*5, np.nan)
                avg_feature += self.element_df.loc[key].values * fractional_composition[key]

            elements_in_formula = list(fractional_composition.keys())
            element_stats = self.element_df.loc[elements_in_formula]
            diff_feature = element_stats.max() - element_stats.min()
            max_feature = element_stats.max()
            min_feature = element_stats.min()
            std_feature = element_stats.std(ddof=0)

            features = np.concatenate([avg_feature, diff_feature, max_feature, min_feature, std_feature])
            return features
        except Exception as e:
            print(f"An error occurred while processing the formula {formula}: {e}")
            return np.full(len(self.element_df.columns) * 5, np.nan)

gf=Vectorize_Formula()

features=[]

for formula in permittivity_df["Composition"]:
    features.append(gf.get_features(formula))

X = pd.DataFrame(features, columns = gf.column_names)
RP_df_first_three = RP_df.iloc[:, :3]
RP_df_rest = RP_df.iloc[:, 3:]
RP_Feature_Set=pd.concat([RP_df_first_three, X, RP_df_rest], axis=1)
RP_Feature_Set.to_excel('RP_feature_set_tester.xlsx', index=False)

### Predict Relative Permittivity

In [ ]:
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold

RPDB = pd.read_excel(r'Training_Set_updated_for_RP.xlsx')
array = RPDB.values
X = array[:,2:100]
Y = array[:,1]
Compd = array[:,0]

RP_model = XGBRegressor(tree_method='hist', device='cuda',  n_estimators=900, learning_rate=0.05,
                          max_depth=4, min_child_weight=6, subsample=0.6, base_score=0.5,
                          colsample_bytree=0.4, colsample_bylevel=1, colsample_bynode=1,
                          reg_alpha=0, reg_lambda=1
                          )
kf = KFold(n_splits=10, shuffle=True, random_state=42)
cv_results = cross_val_score(RP_model, X, Y, cv=kf, scoring='r2')

print(f"\nMean R2: {np.mean(cv_results):.4f} (±{np.std(cv_results):.4f})")

RP_model.fit(X,Y)
prediction = pd.read_excel(r'RP_feature_set_tester.xlsx')
a = prediction.values
b = a[:,3:101]
result = RP_model.predict(b)
composition=prediction['Composition']
result=pd.DataFrame(result, columns=['Predicted RP'])
predicted=np.column_stack((composition,result))
predicted_RP=pd.DataFrame(predicted)
predicted_RP.to_excel('Predicted_RP_tester.xlsx', index=False, header=("Composition","Predicted RP"))

CS_df_left = CS_df.iloc[:, :3]
CS_df_right = CS_df.iloc[:, 3:]
CS_feature_set_df = pd.concat([CS_df_left, result, CS_df_right], axis=1)
CS_feature_set_df.to_excel('CS_feature_set_tester.xlsx', index=False)

d1_df_left = d1_df.iloc[:, :3]
d1_df_right = d1_df.iloc[:, 3:]
d1_with_RP_df = pd.concat([d1_df_left, result, d1_df_right], axis=1)

### Predict Centroid Shift

In [ ]:
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_absolute_error

CSDB = pd.read_excel('Training_Set_updated_for_CS.xlsx')
array = CSDB.values
X = array[:,2:10]
Y = array[:,1]
Compd = array[:,0]

CS_model = XGBRegressor(tree_method='hist', device='cuda', n_estimators=300, learning_rate=0.03,
                           max_depth=9, min_child_weight=2, subsample=0.6,
                           colsample_bytree=0.7, colsample_bylevel=1, colsample_bynode=1,
                           reg_alpha=0, reg_lambda=0)

logo=LeaveOneGroupOut()

all_Y_val = []
all_Y_pred = []
mae_values = []

for train_index, test_index in logo.split(X, Y, Compd):
    X_train, X_test = X[train_index], X[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]

    CS_model.fit(X_train, Y_train)
    Y_pred = CS_model.predict(X_test)

    mae = mean_absolute_error(Y_test, Y_pred)
    mae_values.append(mae)

    all_Y_val.extend(Y_test)
    all_Y_pred.extend(Y_pred)

final_r2 = r2_score(all_Y_val, all_Y_pred)
avg_mae = np.mean(mae_values)

print(f"Mean Absolute Error (MAE): {avg_mae:.4f}")
print(f"R-squared (R2): {final_r2:.4f}")

CS_model.fit(X,Y)
prediction = pd.read_excel(r'CS_feature_set_tester.xlsx')
a = prediction.values
b = a[:,3:11]
result = CS_model.predict(b)
composition=prediction['Composition']
result=pd.DataFrame(result, columns=['Predicted CS'])
predicted=np.column_stack((composition,result))
predicted_CS=pd.DataFrame(predicted)
predicted_CS.to_excel('Predicted_CS_tester.xlsx', index=False, header=("Composition","Predicted CS"))

d1_with_RP_left = d1_with_RP_df.iloc[:, :3]
d1_with_RP_right = d1_with_RP_df.iloc[:, 3:]
d1_feature_set_df = pd.concat([d1_with_RP_left, result, d1_with_RP_right], axis=1)

### Predict 5d1 of Ce3+

In [ ]:
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_absolute_error

class Vectorize_Formula:
    def __init__(self):
        elem_dict = pd.read_excel(r'elements_5d1_new.xlsx') # CHECK NAME OF FILE
        self.element_df = pd.DataFrame(elem_dict)
        self.element_df.set_index('Element',inplace=True)
        self.column_names = []
        for string in ['avg','diff','max','min','std']:
            for column_name in list(self.element_df.columns.values):
                self.column_names.append(f'{string}_{column_name}')

    def get_features(self, formula):
        try:
            fractional_composition = Composition(formula).fractional_composition.as_dict()
            avg_feature = np.zeros(len(self.element_df.columns))

            for key in fractional_composition:
                if key not in self.element_df.index:
                    print('The element:', key, 'from formula', formula,'is not currently supported in our database')
                    return np.full(len(self.element_df.colums)*5, np.nan)
                avg_feature += self.element_df.loc[key].values * fractional_composition[key]

            elements_in_formula = list(fractional_composition.keys())
            element_stats = self.element_df.loc[elements_in_formula]
            diff_feature = element_stats.max() - element_stats.min()
            max_feature = element_stats.max()
            min_feature = element_stats.min()
            std_feature = element_stats.std(ddof=0)

            features = np.concatenate([avg_feature, diff_feature, max_feature, min_feature, std_feature])
            return features
        except Exception as e:
            print(f"An error occurred while processing the formula {formula}: {e}")
            return np.full(len(self.element_df.columns) * 5, np.nan)

gf=Vectorize_Formula()

features=[]

for formula in d1_feature_set_df["Composition"]:
    features.append(gf.get_features(formula))

temp = pd.DataFrame(features, columns = gf.column_names)
columns_to_extract = [0, 2, 3, 4, 5, 8, 13, 20, 23, 28]
comp_feature = temp.iloc[:, columns_to_extract]
completed=pd.concat([d1_feature_set_df, comp_feature], axis=1)
completed.to_excel('5d1_whole_feature_set_tester.xlsx', index=False)

FDODB=pd.read_excel('Training_Set_updated_for_5d1_RFE17.xlsx')
array = FDODB.values
X = array[:,2:]
Y = array[:,1]
Compd = array[:,0]

best_model = XGBRegressor(tree_method='hist', device='cuda', n_estimators=500, learning_rate=0.03,
                          max_depth=8, min_child_weight=8, subsample=0.6, base_score=0.5,
                          colsample_bytree=0.9, colsample_bylevel=1, colsample_bynode=1,
                          reg_alpha=0, reg_lambda=0
                          )

logo=LeaveOneGroupOut()
# Map group labels to integers
unique_groups = np.unique(Compd)
group_to_index = {group: idx for idx, group in enumerate(unique_groups)}

all_Y_val = []
all_Y_pred = []
mae_values = []

for train_index, test_index in logo.split(X, Y, Compd):
    X_train, X_test = X[train_index], X[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]

    CS_model.fit(X_train, Y_train)
    Y_pred = CS_model.predict(X_test)

    mae = mean_absolute_error(Y_test, Y_pred)
    mae_values.append(mae)

    all_Y_val.extend(Y_test)
    all_Y_pred.extend(Y_pred)

final_r2 = r2_score(all_Y_val, all_Y_pred)
avg_mae = np.mean(mae_values)

print(f"Mean Absolute Error (MAE): {avg_mae:.4f}")
print(f"R-squared (R2): {final_r2:.4f}")

best_model.fit(X,Y)

prediction = pd.read_excel(r'5d1_whole_feature_set_tester.xlsx')
a = prediction.values
b = a[:,3:]
result = best_model.predict(b)
composition=prediction['Composition']
result_df=pd.DataFrame(result, columns=['Predicted 5d1 for Ce3+'])
divider = 1.23984193 * 10**3
result_in_nm = result_df['Predicted 5d1 for Ce3+'].map(lambda x: round(divider/x, 2))
result_in_nm.name = 'Pred 5d1 in nm'
predicted=np.column_stack((composition,result,result_in_nm))
predicted_5d1=pd.DataFrame(predicted)
predicted_5d1.to_excel('Predicted_5d1_tester.xlsx', index=False)